# Exercise 14.2: Non-Parametric Permutation Test for AI Evals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-14/02-permutation-test-evals.ipynb)

**Book Reference:** Chapter 14 — Statistical Testing and Experimental Evidence (`chapters/22-statistical-testing.tex`, Section 1)  
**Estimated Time:** 15–20 min

### Learning Objectives
- Implement a two-sided permutation test for comparing two AI evaluation score arrays.
- Compare permutation $p$-values with parametric $z$-test $p$-values on binary evals.
- Apply permutation testing to non-binary (graded) LLM rubric scores without normality assumptions.


In [ ]:
import numpy as np

### Mathematical Context

A **permutation test** makes no parametric distributional assumptions. Under the null hypothesis $H_0$ that Model A and Model B have identical score distributions, the group labels (`A` vs `B`) are exchangeable.

Given arrays $\mathbf{a}$ (length $n_A$) and $\mathbf{b}$ (length $n_B$):
1. Compute the observed difference in means $\Delta_{\text{obs}} = \bar{\mathbf{a}} - \bar{\mathbf{b}}$.
2. Pool all $n_A + n_B$ scores, randomly shuffle them $R$ times, split into sizes $n_A$ and $n_B$, and compute $\Delta_r$.
3. The two-sided $p$-value is the fraction of permutations with $|\Delta_r| \ge |\Delta_{\text{obs}}|$.

In [ ]:
def permutation_test(
    scores_a: np.ndarray,
    scores_b: np.ndarray,
    num_permutations: int = 5000,
    seed: int = 42,
) -> tuple[float, float]:
    """Run a two-sided permutation test comparing mean(scores_a) vs mean(scores_b).

    Args:
        scores_a: 1D NumPy array of evaluation scores for Model A.
        scores_b: 1D NumPy array of evaluation scores for Model B.
        num_permutations: Number of random label permutations (default 5000).
        seed: Random seed (default 42).

    Returns:
        Tuple (obs_diff, p_value) where obs_diff = mean(scores_a) - mean(scores_b)
        and p_value = fraction of permutations with |perm_diff| >= |obs_diff| - 1e-12.
    """
    # TODO: Pool scores, draw num_permutations random shuffles with rng = np.random.default_rng(seed),
    # compute each perm_diff, and return (obs_diff, p_value).
    raise NotImplementedError("Complete this TODO")

<details>
<summary>Hint 1: Fast mean difference in permutations</summary>

Notice that if `sum_a = np.sum(perm[:n_a])`, then `sum_b = total_sum - sum_a`, so you don't even need to slice or sum the second half of `perm`! Subtracting `1e-12` from `abs(obs_diff)` avoids floating-point rounding misses when a permutation reproduces the exact observed split.

</details>

In [ ]:
# Self-check assertions
# 1. Binary evals: Model A (84/100) vs Model B (80/100)
a_bin = np.array([1] * 84 + [0] * 16, dtype=float)
b_bin = np.array([1] * 80 + [0] * 20, dtype=float)
diff_bin, p_bin = permutation_test(a_bin, b_bin, num_permutations=3000, seed=42)
assert abs(diff_bin - 0.04) < 1e-12
assert 0.35 < p_bin < 0.60, f"Expected p ~ 0.46, got {p_bin}"

# 2. Graded LLM rubric scores (1-5 scale) with clear improvement
a_rubric = np.array([4.5, 4.8, 4.2, 5.0, 4.6, 4.7, 4.9, 4.3, 4.6, 4.8] * 3)
b_rubric = np.array([3.5, 3.8, 3.2, 4.0, 3.6, 3.7, 3.9, 3.3, 3.6, 3.8] * 3)
diff_rub, p_rub = permutation_test(a_rubric, b_rubric, num_permutations=3000, seed=42)
assert abs(diff_rub - 1.0) < 1e-12
assert p_rub < 0.005, f"Expected very small p-value, got {p_rub}"

print(f"Binary eval (84/100 vs 80/100): diff={diff_bin:+.3f}, perm_p={p_bin:.4f}")
print(f"Rubric eval (clear gap):        diff={diff_rub:+.3f}, perm_p={p_rub:.4f}")
print("All assertions passed!")

### Think Deeper

1. When comparing two LLMs on the **exact same** 100 test prompts, why is a **paired** test (permuting within each prompt, or McNemar's test) often more powerful than an unpaired two-sample test?
2. Why is a permutation test especially useful for skewed metrics like latency or graded LLM-as-a-judge rubric scores?